# Reviewer Figure: SFTF analysis and relation to depth

- **A**: Experimental setup schematic: head-fixed mouse on a running wheel under two-photon imaging during presentation of drifting sinusoidal gratings across panoramic surround monitors (no optic flow or locomotion cue).
- **B**: Depth tuning curve of example depth-selective neuron (PZAH10.2d_S20230922, ROI 558; preferred virtual depth = 0.70 m): Measured mean $\Delta F/F$ across 8 virtual depths during closed-loop locomotion (blue dots: mean $\pm$ SEM; blue curve: Gaussian fit).
- **C**: Example depth-selective neuron SFTF responses (PZAH10.2d_S20230922, ROI 558): Measured mean $\Delta F/F$ responses across spatial and temporal frequencies for each of the 8 drift directions surrounding the polar direction tuning curve (blue shading).
- **D**: Distribution of direction selectivity index (DSI) for depth-selective neurons ($n = 390$, black step) versus not depth-selective neurons ($n = 1052$, grey bars; Mann-Whitney $U$ test $p = 0.94$).
- **E**: Polar histogram of preferred direction for direction-selective neurons ($\text{DSI} > 0.6$) among depth-selective ($n = 137$, black step outline) and not depth-selective neurons ($n = 363$, grey filled bars). Crimson arrow indicates the direction of visual optic flow ($180^\circ$).
- **F**: Preferred grating speed vs. preferred virtual depth across SF/TF-tuned, depth-selective neurons ($n = 390$, Spearman $\rho = -0.16$, 95% CI [-0.29, -0.05], hierarchical bootstrap $p = 0.0069$; crimson line: linear regression in log-log space with 95% confidence band).
- **G**: Preferred temporal frequency vs. preferred virtual depth ($n = 390$, Spearman $\rho = 0.06$, 95% CI [-0.15, 0.18], hierarchical bootstrap $p = 0.566$; not significant, no regression line).
- **H**: Preferred spatial frequency vs. preferred virtual depth ($n = 390$, Spearman $\rho = 0.36$, 95% CI [0.13, 0.46], hierarchical bootstrap $p = 0.0097$; crimson line: linear regression in log-log space with 95% confidence band).
- **I**: Preferred optic flow speed vs. preferred spatial frequency across neurons tuned to both SFTF and RS-OF ($n = 426$, Spearman $\rho = -0.39$, 95% CI [-0.49, -0.20], hierarchical bootstrap $p < 0.001$; crimson line: linear regression in log-log space with 95% confidence band).
- **J**: Example depth-selective neurons temporal frequency tuning curves across static locomotion (black solid line) and 4 running-speed bins (5.0–11.3, 11.3–22.6, 22.6–45.2, 45.2–90.5 cm/s; devon colormap). Error bars indicate SEM across trial presentations.
- **K**: Example depth-selective neurons spatial frequency tuning curves across static locomotion (black solid line) and 4 running-speed bins (acton colormap). Error bars indicate SEM across trial presentations.
- **L**: Population average temporal frequency tuning curve relative to peak (octaves) evaluated on held-out trials across 4 running-speed bins and static locomotion for depth-selective neurons ( = 146$; dots: mean $\pm$ 95% bootstrap CI; devon colormap, static in black solid line).
- **M**: Population average spatial frequency tuning curve relative to peak (octaves) evaluated on held-out trials across 4 running-speed bins and static locomotion for depth-selective neurons ( = 175$; dots: mean $\pm$ 95% bootstrap CI; acton colormap, static in black solid line).
- **N**: Population average temporal frequency tuning curve where each condition is min-max normalised to its own dynamic range (0 to 1; devon colormap).
- **O**: Population average spatial frequency tuning curve where each condition is min-max normalised to its own dynamic range (0 to 1; acton colormap).

In [ ]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
from functools import partial
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from PIL import Image
import flexiznam as flz

from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM
from cottage_analysis.analysis import common_utils
from cottage_analysis.preprocessing import synchronisation
from cottage_analysis.analysis.fit_gaussian_blob import GratingParams, grating_tuning
from v1_depth_map.figure_utils.rf_distance import load_session_neurons
from v1_depth_map.paths import get_figures_roots

style.setup_figure_fonts()

PROJECT = "hey2_3d-vision_foodres_20220101"
SESSIONS = [
    "PZAH10.2d_S20230920",
    "PZAH10.2d_S20230922",
    "PZAH10.2f_S20230817",
    "PZAH10.2f_S20230912",
    "PZAH10.2f_S20230914",
    "PZAH10.2f_S20230924",
]
PHOTODIODE_PROTOCOL = 5  # all PZAH mice
TRACES = {"anatomical_only": 3, "ast_neuropil": False}
PARAMS = ["SpatialFrequency", "TemporalFrequency", "Angle"]
LAG = 8  # onset volumes skipped (0.53 s at 15.2 Hz; window ends at the next onset, as gratings.generate_trials_df)
FRAME_LAG = 0  # frames between stimulus / RS and dF/F in the frame-level analyses (as the RS/OF fits)
MIN_VOLUMES = 10  # drop trials shorter than this (truncated at the end of a recording)
P_RESPONSIVE = 0.01  # ANOVA across conditions
P_SELECTIVE = 0.05  # gOSI / gDSI shuffle
N_SHUFFLES = 1000
REBUILD = False  # True: re-sync the recordings instead of reading the trial cache
# selection criteria and stats of the thesis SFTF figure (yiran-phd-thesis/yh_thesis/depth/SFTF.ipynb),
# except SF/TF tuned, which is cross-validated (one-sided Spearman p < 0.05, as for depth)
# RS/OF tuned (panels B, C): rsof_test_rsq_closedloop_g2d above the empirical-null threshold of figure 2
# (mirrored negative tail, common_utils.add_rsq_significance)
RSOF_SIG_PERCENTILE = 95
DSI_THR = 0.6  # panel F: direction-selective cells only
N_BOOTS = 20000  # hierarchical bootstrap (mouse, session) of the B, C correlations (exported stats)
N_BOOTS_EXPLORATORY = 2000  # exploratory sections (not exported): p resolution 5e-4

GROUPS = {"depth tuned": "#2a78d6", "not depth tuned": "#eb6834"}

flm = flz.get_flexilims_session(PROJECT)
_, SAVE_ROOT = get_figures_roots(flm, fig_subdir="fig_reviewer_gratings")
CACHE = SAVE_ROOT / "cache"
FRAMES_DIR = SAVE_ROOT / "running_models" / "frames"  # frame-level data of the running-speed models
CACHE.mkdir(parents=True, exist_ok=True)
print(f"Saving to {SAVE_ROOT}")

In [ ]:
# Load data and precomputed statistics for figure
csv_path = SAVE_ROOT / "grating_tuning_cells.csv"
cells = pd.read_csv(csv_path)

depth_sftf = cells[cells.depth_tuned & cells.sftf_tuned].reset_index(drop=True)

tuned = cells[cells.sftf_tuned & cells.depth_tuned]
tuned_other = cells[cells.sftf_tuned & ~cells.depth_tuned]

DSI_THR = 0.6
dir_tuned = tuned[tuned.fit_dsi > DSI_THR]
dir_tuned_other = tuned_other[tuned_other.fit_dsi > DSI_THR]

# Load example cell raw trials from cache (PZAH10.2d_S20230922 ROI 558: preferred depth 0.70 m)
example_session = "PZAH10.2d_S20230922"
example_roi = 558
trials_cache = CACHE / f"{example_session}_sftf_trials.pickle"
trials_data = pd.read_pickle(trials_cache)

trials_df = trials_data["trials"].copy()
trials_df["response"] = trials_data["dff"][:, example_roi]

angle_range = np.sort(trials_df.Angle.unique())
sf_range = np.sort(trials_df.SpatialFrequency.unique())
tf_range = np.sort(trials_df.TemporalFrequency.unique())

all_angles = np.zeros((len(angle_range), len(sf_range), len(tf_range)))
for i, angle in enumerate(angle_range):
    this_df = trials_df[trials_df.Angle == angle]
    for isf, sf in enumerate(sf_range):
        for itf, tf in enumerate(tf_range):
            all_angles[i, isf, itf] = this_df[
                (this_df.SpatialFrequency == sf) & (this_df.TemporalFrequency == tf)
            ]["response"].mean()

vmax = np.round(np.nanmax(all_angles), 1)

# Load example cell depth tuning data
depth_example_cache = CACHE / f"{example_session}_roi{example_roi}_depth_example.pickle"
if depth_example_cache.exists():
    depth_example_data = pd.read_pickle(depth_example_cache)
    example_depth_trials = depth_example_data["trials_depth"]
    example_depth_neurons = depth_example_data["neurons_df"].copy()
    example_depth_neurons.index = [0]
else:
    from v1_depth_map.figure_utils import sftf

    example_depth_trials = sftf.load_depth_trials(flm, example_session)
    example_depth_neurons, _ = load_session_neurons(flm, example_session)

import contextlib, io


def bootstrap_spearman(x, y, sessions, n_boots=N_BOOTS):
    # Hierarchical bootstrap resampled on session then cells
    df_boot = pd.DataFrame(
        dict(x=np.asarray(x), y=np.asarray(y), session=np.asarray(sessions))
    )
    df_boot = df_boot[np.isfinite(df_boot.x) & np.isfinite(df_boot.y)].reset_index(
        drop=True
    )
    with contextlib.redirect_stderr(io.StringIO()):
        rho, dist = common_utils.hierarchical_bootstrap_stats(
            df_boot,
            n_boots,
            ["x"],
            ["session"],
            ycol=["y"],
            correlation=True,
            plot=False,
        )
    return dict(
        n=len(df_boot),
        rho=float(rho[0]),
        ci=np.percentile(dist[:, 0], [2.5, 97.5]),
        p=float(common_utils.calculate_pval_from_bootstrap(dist[:, 0], value=0)),
    )


# Precompute / load bootstrap statistics for F, G, H
stat_sf_depth = bootstrap_spearman(
    np.exp(depth_sftf.fit_sf0), depth_sftf.preferred_depth * 100, depth_sftf.session
)
stat_tf_depth = bootstrap_spearman(
    np.exp(depth_sftf.fit_tf0), depth_sftf.preferred_depth * 100, depth_sftf.session
)
stat_speed_depth = bootstrap_spearman(
    depth_sftf.pref_speed, depth_sftf.preferred_depth * 100, depth_sftf.session
)

print(
    f"SF vs depth: n = {stat_sf_depth['n']}, rho = {stat_sf_depth['rho']:.3f}, 95% CI {stat_sf_depth['ci'].round(3)}, p = {stat_sf_depth['p']:.4f}"
)
print(
    f"TF vs depth: n = {stat_tf_depth['n']}, rho = {stat_tf_depth['rho']:.3f}, 95% CI {stat_tf_depth['ci'].round(3)}, p = {stat_tf_depth['p']:.4f}"
)
print(
    f"Speed vs depth: n = {stat_speed_depth['n']}, rho = {stat_speed_depth['rho']:.3f}, 95% CI {stat_speed_depth['ci'].round(3)}, p = {stat_speed_depth['p']:.4f}"
)

# Cells tuned to both SFTF and RS-OF (g2d) for Panel I (OF vs SF)
both = cells[cells.sftf_tuned & cells.rsof_tuned].reset_index(drop=True)
stat_sf_of = bootstrap_spearman(
    np.exp(both.fit_sf0), both.preferred_of, both.session
)
print(
    f"SF vs OF: n = {stat_sf_of['n']}, rho = {stat_sf_of['rho']:.3f}, 95% CI {stat_sf_of['ci'].round(3)}, p = {stat_sf_of['p']:.4f}"
)

# Held-out population TF / SF curves per running-speed bin, still frames as bin -1 (panels L-O), cached
from v1_depth_map.figure_utils import sftf

summary_data = sftf.build_speed_curves(SAVE_ROOT)
heldout_means = summary_data["heldout_means"]
mean_peaks = summary_data["mean_peaks"]
speed_labels = summary_data["speed_labels"]

SPEED_EDGES_CM_S = [5, 11.31, 22.63, 45.25, 90.51]  # first bin from 5 cm/s, the still threshold (STILL_THR)
TF_SELECT_THRESHOLD = 0.5
STILL_DIR = SAVE_ROOT / "running_models" / "frames_still"

# 2 exemplar depth-selective cells showing clear rightward shifts in preferred TF with locomotion
EXAMPLE_CELLS_SPEED = [
    ("PZAH10.2f_S20230924", 95, "Cell 1"),
    ("PZAH10.2d_S20230922", 562, "Cell 2"),
]



In [ ]:
# Generate Full SFTF Analysis Figure (Panels A to L)
from cottage_analysis.plotting import depth_selectivity_plots
from v1_depth_map.figure_utils import sftf
from scipy.stats import mannwhitneyu, t

schematic_path = Path(
    "/Users/blota/Documents/code/v1_depth_map/v1_depth_map/figures/schematic_gratings.png"
)
if not schematic_path.exists():
    schematic_path = SAVE_ROOT / "schematic_gratings.png"


# Helper for polar histogram
def polar_hist(ax, values_deg, period, n_bins, step=False, color="k", proportion=True, lw=1):
    width = period / n_bins
    counts, _ = np.histogram(
        (values_deg + width / 2) % period, bins=n_bins, range=(0, period)
    )
    heights = counts / counts.sum() if proportion else counts
    if step:
        theta = np.linspace(0, period, 721)
        r = heights[((theta + width / 2) // width).astype(int) % n_bins]
        ax.plot(np.deg2rad(theta), r, color=color, lw=lw, zorder=3)
    else:
        centres = np.deg2rad(np.arange(n_bins) * width)
        ax.bar(centres, heights, width=np.deg2rad(width) * 0.95, color=color, lw=0)
        if period < 360:
            ax.bar(np.deg2rad(period), heights[0], width=np.deg2rad(width) * 0.95, color=color, lw=0)
    ax.set_thetamin(0)
    ax.set_thetamax(period)
    ax.set_xticks(np.deg2rad(np.arange(0, period + 1 if period < 360 else period, 45)))
    ax.set_rlabel_position(period / n_bins / 2)
    ax.yaxis.set_major_locator(plt.MaxNLocator(3))
    ax.tick_params(labelsize=5.0)
    return heights


def fit_line_with_ci(x, y, n_points=100):
    valid = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x, y = x[valid], y[valid]
    lx, ly = np.log(x), np.log(y)
    n = len(lx)
    p = np.polyfit(lx, ly, 1)
    x_grid = np.geomspace(x.min(), x.max(), n_points)
    lx_grid = np.log(x_grid)
    ly_fit = np.polyval(p, lx_grid)
    residuals = ly - np.polyval(p, lx)
    s_err = np.sqrt(np.sum(residuals**2) / (n - 2))
    x_mean = np.mean(lx)
    ss_x = np.sum((lx - x_mean) ** 2)
    se_line = s_err * np.sqrt(1.0 / n + (lx_grid - x_mean) ** 2 / ss_x)
    t_val = t.ppf(0.975, df=n - 2)
    y_fit = np.exp(ly_fit)
    y_lo = np.exp(ly_fit - t_val * se_line)
    y_hi = np.exp(ly_fit + t_val * se_line)
    return x_grid, y_fit, y_lo, y_hi


# Figure dimensions: full width 18 cm, height 18.5 cm
FIG_W_CM = 18.0
FIG_H_CM = 18.5
fig = plt.figure(figsize=(FIG_W_CM * CM, FIG_H_CM * CM))

# Font size definitions matching rest of notebooks
FS_PANEL = 8.0
FS_TITLE = 6.5
FS_LABEL = 6.5
FS_TICK = 5.0
FS_LEGEND = 5.0

# Coordinates in cm converted to figure fraction
# Row 1: y from 14.0 to 17.8 cm (h = 3.8 cm)
y_row1 = 14.0 / FIG_H_CM
h_row1 = 3.8 / FIG_H_CM

# Row 2: y from 9.3 to 12.3 cm (h = 3.0 cm, aspect ratio ~1:1 with w = 2.85 cm)
y_row2 = 9.3 / FIG_H_CM
h_row2 = 3.0 / FIG_H_CM

# Row 3: y from 5.2 to 8.0 cm (h = 2.8 cm)
y_row3 = 5.2 / FIG_H_CM
h_row3 = 2.8 / FIG_H_CM

# Row 4: y from 1.1 to 3.9 cm (h = 2.8 cm)
y_row4 = 1.1 / FIG_H_CM
h_row4 = 2.8 / FIG_H_CM

# ----------------- PANEL A: Schematic -----------------
w_a = 2.5 / FIG_W_CM
h_a = 1.8 / FIG_H_CM
x_a = 0.035
y_a = y_row1 + h_row1 - h_a

ax_a = fig.add_axes([x_a, y_a, w_a, h_a])
img = Image.open(schematic_path)
ax_a.imshow(img)
ax_a.axis("off")
fig.text(x_a - 0.02, y_row1 + h_row1 + 0.015, "A", fontweight="bold", fontsize=FS_PANEL, va="bottom")

# ----------------- PANEL B: Example Depth Tuning -----------------
w_b = 2.8 / FIG_W_CM
h_b = 1.55 / FIG_H_CM
x_b = x_a
y_b = y_row1 + 0.35 / FIG_H_CM
ax_b = fig.add_axes([x_b, y_b, w_b, h_b])
depth_selectivity_plots.plot_depth_tuning_curve(
    neurons_df=example_depth_neurons,
    trials_df=example_depth_trials,
    roi=0,
    rs_thr=None,
    plot_fit=True,
    plot_smooth=False,
    linewidth=1.2,
    linecolor="#2a78d6",
    closed_loop=1,
    fontsize_dict={"title": FS_TITLE, "label": FS_LABEL, "tick": FS_TICK},
    markersize=2.5,
    marker="o",
    markeredgecolor="#2a78d6",
    ax=ax_b,
)
for line in ax_b.lines:
    if line.get_linestyle() == "None":
        line.set_marker("o")
        line.set_markersize(2.5)
ax_b.set_title(f"Example depth tuning (ROI {example_roi})", fontsize=FS_TITLE, pad=2)
fig.text(x_b - 0.02, y_b + h_b + 0.015, "B", fontweight="bold", fontsize=FS_PANEL, va="bottom")

# ----------------- PANEL C: Example Cell SFTF Raw Responses -----------------
x_c = 0.225
w_c = 3.6 / FIG_W_CM
fig.text(x_c - 0.025, y_row1 + h_row1 + 0.015, "C", fontweight="bold", fontsize=FS_PANEL, va="bottom")

grid_map = {
    (0, 0): 135, (0, 1): 90, (0, 2): 45,
    (1, 0): 180,             (1, 2): 0,
    (2, 0): 225, (2, 1): 270, (2, 2): 315,
}

sub_w = w_c / 3.3
sub_h = h_row1 / 3.3
gap_x = (w_c - 3 * sub_w) / 2
gap_y = (h_row1 - 3 * sub_h) / 2

polar_ax = fig.add_axes(
    [x_c + sub_w + gap_x + 0.002, y_row1 + sub_h + gap_y + 0.002, sub_w * 0.82, sub_h * 0.82],
    projection="polar",
)
dir_tuning = np.max(all_angles, axis=(1, 2))
dir_tuning = np.append(dir_tuning, dir_tuning[0])
dirs = np.deg2rad(np.append(angle_range, angle_range[0]))

polar_ax.plot(dirs, dir_tuning, color="k", linewidth=0.8)
polar_ax.fill_between(dirs, dir_tuning, color="#4A90E2", alpha=0.7)
polar_ax.set_ylim(0, vmax)
polar_ax.set_yticks([0, vmax])
polar_ax.set_yticklabels(["0", f"{vmax:g}"], fontsize=FS_TICK)
polar_ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
polar_ax.set_xticklabels([f"{int(d)}°" for d in [0, 45, 90, 135, 180, 225, 270, 315]], fontsize=4.5)
polar_ax.tick_params(axis="x", pad=-3)

last_im = None
for row in range(3):
    for col in range(3):
        if (row, col) == (1, 1):
            continue
        angle = grid_map[(row, col)]
        angle_idx = np.where(angle_range == angle)[0][0]

        ax_x = x_c + col * (sub_w + gap_x)
        ax_y = y_row1 + (2 - row) * (sub_h + gap_y)
        ax_dir = fig.add_axes([ax_x, ax_y, sub_w, sub_h])

        mat = all_angles[angle_idx].T
        im_plot = ax_dir.imshow(
            mat,
            origin="lower",
            vmax=vmax,
            vmin=0,
            cmap="magma",
            aspect="auto",
        )
        last_im = im_plot

        if col == 0:
            ax_dir.set_yticks(range(len(tf_range)))
            ax_dir.set_yticklabels([f"{t:g}" for t in tf_range], fontsize=FS_TICK)
            if row == 1:
                ax_dir.set_ylabel("TF (Hz)", fontsize=FS_LABEL)
        else:
            ax_dir.set_yticks([])
            ax_dir.set_yticklabels([])

        if row == 2:
            ax_dir.set_xticks(range(len(sf_range)))
            ax_dir.set_xticklabels([f"{s:g}" for s in sf_range], fontsize=FS_TICK, rotation=90)
            ax_dir.tick_params(axis="x", pad=1)
            if col == 1:
                ax_dir.set_xlabel("SF (cpd)", fontsize=FS_LABEL, labelpad=2)
        else:
            ax_dir.set_xticks([])
            ax_dir.set_xticklabels([])

cax = fig.add_axes([x_c + w_c + 0.012, y_row1 + gap_y, 0.006, h_row1 - 2 * gap_y])
cbar = fig.colorbar(last_im, cax=cax)
cbar.set_ticks([0, vmax])
cbar.ax.tick_params(labelsize=FS_TICK)
cbar.ax.set_title("ΔF/F", fontsize=FS_LABEL, pad=3)

# ----------------- PANEL D: DSI Distribution -----------------
x_d = 0.495
w_d = 3.5 / FIG_W_CM
h_d = 3.3 / FIG_H_CM
y_d = y_row1 + (h_row1 - h_d) / 2
ax_d = fig.add_axes([x_d, y_d, w_d, h_d])

dsi_bins = np.linspace(0, 1, 21)
ax_d.hist(
    tuned_other.fit_dsi.dropna(),
    bins=dsi_bins,
    weights=np.full(len(tuned_other.fit_dsi.dropna()), 1 / len(tuned_other.fit_dsi.dropna())),
    color="lightgrey",
    label=f"other (n = {len(tuned_other.fit_dsi.dropna())})",
)
ax_d.hist(
    tuned.fit_dsi.dropna(),
    bins=dsi_bins,
    weights=np.full(len(tuned.fit_dsi.dropna()), 1 / len(tuned.fit_dsi.dropna())),
    histtype="step",
    color="k",
    lw=1.5,
    label=f"depth selective (n = {len(tuned.fit_dsi.dropna())})",
)
mwu = mannwhitneyu(tuned.fit_dsi.dropna(), tuned_other.fit_dsi.dropna())
ax_d.set_xlabel("Direction selectivity index", fontsize=FS_LABEL)
ax_d.set_ylabel("Proportion of neurons", fontsize=FS_LABEL)
ax_d.tick_params(labelsize=FS_TICK)
ax_d.legend(fontsize=FS_LEGEND, frameon=False, loc="upper right")
ax_d.spines[["top", "right"]].set_visible(False)
ax_d.set_title(f"Direction selectivity\nMann-Whitney p = {mwu.pvalue:.2f}", fontsize=FS_TITLE)
fig.text(x_d - 0.03, y_row1 + h_row1 + 0.015, "D", fontweight="bold", fontsize=FS_PANEL, va="bottom")

# ----------------- PANEL E: Preferred Direction Polar Plot -----------------
x_e = 0.770
w_e = 3.0 / FIG_W_CM
h_e = 3.0 / FIG_H_CM
y_e = y_row1 + (h_row1 - h_e) / 2 - 0.01
ax_e = fig.add_axes([x_e, y_e, w_e, h_e], projection="polar")

polar_hist(ax_e, dir_tuned_other.fit_dir.values, 360, 8, color="lightgrey")
polar_hist(ax_e, dir_tuned.fit_dir.values, 360, 8, step=True, color="k", lw=1.5)

# Optic flow direction arrow toward 180°
ax_e.annotate(
    "",
    xy=(np.pi, 0.155),
    xytext=(np.pi, 0.0),
    arrowprops=dict(arrowstyle="->", color="crimson", lw=1.3, mutation_scale=10),
    zorder=5,
)
ax_e.text(
    np.pi, 0.08, "Optic Flow",
    fontsize=5.0, color="crimson", fontweight="bold",
    ha="center", va="bottom", zorder=6
)

ax_e.set_title(
    f"Preferred direction (DSI > {DSI_THR})\nblack: depth (n={len(dir_tuned)}), grey: other (n={len(dir_tuned_other)})",
    fontsize=FS_TITLE,
    pad=8,
)
fig.text(x_e - 0.055, y_row1 + h_row1 + 0.015, "E", fontweight="bold", fontsize=FS_PANEL, va="bottom")

# ----------------- ROW 2: SCATTERS (F, G, H, I) -----------------
# 4 panels: 1) Depth vs Grating Speed, 2) Depth vs TF, 3) Depth vs SF, 4) OF vs SF
scatter_kwargs = dict(s=7, color="k", alpha=0.35, linewidths=0, zorder=2)
w_scatter = 2.85 / FIG_W_CM
gap_scatter = 1.45 / FIG_W_CM
x_f = 0.055
x_g = x_f + w_scatter + gap_scatter
x_h = x_g + w_scatter + gap_scatter
x_i = x_h + w_scatter + gap_scatter

# PANEL F: Depth vs Grating Speed (Significant - plot line + 95% CI)
ax_f = fig.add_axes([x_f, y_row2, w_scatter, h_row2])
ax_f.scatter(depth_sftf.pref_speed, depth_sftf.preferred_depth * 100, **scatter_kwargs)
x_grid, y_fit, y_lo, y_hi = fit_line_with_ci(depth_sftf.pref_speed, depth_sftf.preferred_depth * 100)
ax_f.fill_between(x_grid, y_lo, y_hi, color="crimson", alpha=0.2, zorder=3)
ax_f.plot(x_grid, y_fit, color="crimson", lw=1.2, zorder=4)
ax_f.set_xscale("log")
ax_f.set_yscale("log")
ax_f.set_xlabel("Preferred grating speed (deg/s)", fontsize=FS_LABEL)
ax_f.set_ylabel("Preferred virtual depth (cm)", fontsize=FS_LABEL, labelpad=2)
ax_f.tick_params(labelsize=FS_TICK)
ax_f.spines[["top", "right"]].set_visible(False)
ci = stat_speed_depth["ci"]
ax_f.set_title(
    f"n = {stat_speed_depth['n']}, ρ = {stat_speed_depth['rho']:.2f} [{ci[0]:.2f}, {ci[1]:.2f}]\nbootstrap p = {stat_speed_depth['p']:.4f}",
    fontsize=FS_TITLE,
)
fig.text(x_f - 0.045, y_row2 + h_row2 + 0.015, "F", fontweight="bold", fontsize=FS_PANEL, va="bottom")

# PANEL G: Depth vs TF (NOT significant - NO trend line)
ax_g = fig.add_axes([x_g, y_row2, w_scatter, h_row2])
ax_g.scatter(np.exp(depth_sftf.fit_tf0), depth_sftf.preferred_depth * 100, **scatter_kwargs)
ax_g.set_xscale("log")
ax_g.set_yscale("log")
ax_g.set_xlabel("Preferred temporal frequency (Hz)", fontsize=FS_LABEL)
ax_g.set_ylabel("Preferred virtual depth (cm)", fontsize=FS_LABEL, labelpad=2)
ax_g.tick_params(labelsize=FS_TICK)
ax_g.spines[["top", "right"]].set_visible(False)
ci = stat_tf_depth["ci"]
ax_g.set_title(
    f"n = {stat_tf_depth['n']}, ρ = {stat_tf_depth['rho']:.2f} [{ci[0]:.2f}, {ci[1]:.2f}]\nbootstrap p = {stat_tf_depth['p']:.3f}",
    fontsize=FS_TITLE,
)
fig.text(x_g - 0.045, y_row2 + h_row2 + 0.015, "G", fontweight="bold", fontsize=FS_PANEL, va="bottom")

# PANEL H: Depth vs SF (Significant - plot line + 95% CI)
ax_h = fig.add_axes([x_h, y_row2, w_scatter, h_row2])
ax_h.scatter(np.exp(depth_sftf.fit_sf0), depth_sftf.preferred_depth * 100, **scatter_kwargs)
x_grid, y_fit, y_lo, y_hi = fit_line_with_ci(np.exp(depth_sftf.fit_sf0), depth_sftf.preferred_depth * 100)
ax_h.fill_between(x_grid, y_lo, y_hi, color="crimson", alpha=0.2, zorder=3)
ax_h.plot(x_grid, y_fit, color="crimson", lw=1.2, zorder=4)
ax_h.set_xscale("log")
ax_h.set_yscale("log")
ax_h.set_xlabel("Preferred spatial frequency (cpd)", fontsize=FS_LABEL)
ax_h.set_ylabel("Preferred virtual depth (cm)", fontsize=FS_LABEL, labelpad=2)
ax_h.tick_params(labelsize=FS_TICK)
ax_h.spines[["top", "right"]].set_visible(False)
ci = stat_sf_depth["ci"]
ax_h.set_title(
    f"n = {stat_sf_depth['n']}, ρ = {stat_sf_depth['rho']:.2f} [{ci[0]:.2f}, {ci[1]:.2f}]\nbootstrap p = {stat_sf_depth['p']:.4f}",
    fontsize=FS_TITLE,
)
fig.text(x_h - 0.045, y_row2 + h_row2 + 0.015, "H", fontweight="bold", fontsize=FS_PANEL, va="bottom")

# PANEL I: OF vs SF (Significant - plot line + 95% CI)
ax_i = fig.add_axes([x_i, y_row2, w_scatter, h_row2])
x_sf_of = np.exp(both.fit_sf0)
y_sf_of = both.preferred_of
ax_i.scatter(x_sf_of, y_sf_of, **scatter_kwargs)
x_grid, y_fit, y_lo, y_hi = fit_line_with_ci(x_sf_of, y_sf_of)
ax_i.fill_between(x_grid, y_lo, y_hi, color="crimson", alpha=0.2, zorder=3)
ax_i.plot(x_grid, y_fit, color="crimson", lw=1.2, zorder=4)
ax_i.set_xscale("log")
ax_i.set_yscale("log")
ax_i.set_xlabel("Preferred spatial frequency (cpd)", fontsize=FS_LABEL)
ax_i.set_ylabel("Preferred optic flow (deg/s)", fontsize=FS_LABEL, labelpad=2)
ax_i.tick_params(labelsize=FS_TICK)
ax_i.spines[["top", "right"]].set_visible(False)
ci = stat_sf_of["ci"]
p_str = f"{stat_sf_of['p']:.4f}" if stat_sf_of['p'] >= 0.0001 else "< 0.001"
ax_i.set_title(
    f"n = {stat_sf_of['n']}, ρ = {stat_sf_of['rho']:.2f} [{ci[0]:.2f}, {ci[1]:.2f}]\nbootstrap p = {p_str}",
    fontsize=FS_TITLE,
)
fig.text(x_i - 0.045, y_row2 + h_row2 + 0.015, "I", fontweight="bold", fontsize=FS_PANEL, va="bottom")

# ----------------- ROW 3 & 4: RUNNING SPEED TUNING (J, K, L, M, N, O) -----------------
# Taking whole figure width: Example cells (left), Legend (mid-left), Population Col 1 & 2 (mid-right & right)
w_ex = 1.95 / FIG_W_CM
gap_ex = 0.45 / FIG_W_CM
x_ex_start = 0.050

# Load frame data for example cells including Static
ex_binned = {}
for session, roi, _ in EXAMPLE_CELLS_SPEED:
    d = np.load(FRAMES_DIR / f"{session}.npz")
    ds = np.load(STILL_DIR / f"{session}.npz")
    col = int(np.flatnonzero(d["rois"] == roi)[0])
    binned_run = sftf.speed_binned_responses(
        d["X"], d["dff"][:, col], float(d["log_rs_median"]), edges_cm_s=SPEED_EDGES_CM_S, recording=d["recording"]
    )
    stim_still = ds["X"][:3].round(4)
    new_still = np.r_[True, (np.diff(stim_still, axis=1) != 0).any(axis=0) | (ds["recording"][1:] != ds["recording"][:-1])]
    trial_still = np.cumsum(new_still) + 1000000
    still_df = pd.DataFrame({
        "SpatialFrequency": np.exp(ds["X"][0]).round(4),
        "TemporalFrequency": np.exp(ds["X"][1]).round(4),
        "Angle": np.rad2deg(ds["X"][2]).round().astype(int),
        "trial": trial_still,
        0: ds["dff"][:, col],
    })
    ex_binned[(session, roi)] = [("Static", still_df)] + binned_run

# Colormaps: devon for TF, acton for SF (from cmcrameri, cropped to maintain contrast), Static in black
from cmcrameri import cm
devon_colors = cm.devon(np.linspace(0.25, 0.78, 4))
acton_colors = cm.acton(np.linspace(0.20, 0.78, 4))
run_labels = [b[0] for b in binned_run]

def plot_curves_with_static(ax, curves, variable="tf", ylabel="ΔF/F", xlabel=None):
    freq_col = "TemporalFrequency" if variable == "tf" else "SpatialFrequency"
    colors = devon_colors if variable == "tf" else acton_colors
    # Plot Static first (continuous line)
    c_stat = curves[curves["bin"] == "Static"]
    if len(c_stat) > 0:
        ax.errorbar(
            c_stat[freq_col], c_stat["mean"], yerr=c_stat["sem"],
            color="k", ls="-", lw=1.1, marker="o", ms=2.5, label="Static"
        )
    # Plot 4 running bins
    for idx, r_lbl in enumerate(run_labels):
        c_r = curves[curves["bin"] == r_lbl]
        if len(c_r) > 0:
            ax.errorbar(
                c_r[freq_col], c_r["mean"], yerr=c_r["sem"],
                color=colors[idx], ls="-", lw=1.1, marker="o", ms=2.5, label=r_lbl
            )
    ax.set_xscale("log")
    vals = np.sort(curves[freq_col].unique())
    ax.set_xticks(vals)
    ax.set_xticklabels([f"{v:g}" for v in vals], fontsize=FS_TICK)
    ax.minorticks_off()
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=FS_LABEL)
    ax.set_ylabel(ylabel, fontsize=FS_LABEL)
    ax.tick_params(labelsize=FS_TICK)
    ax.spines[["top", "right"]].set_visible(False)

# PANEL J: Example cells TF tuning by running speed (Row 3, left)
fig.text(x_ex_start - 0.030, y_row3 + h_row3 + 0.015, "J", fontweight="bold", fontsize=FS_PANEL, va="bottom")
for j, (session, roi, label) in enumerate(EXAMPLE_CELLS_SPEED):
    ax_ij = fig.add_axes([x_ex_start + j * (w_ex + gap_ex), y_row3, w_ex, h_row3])
    ex_popt = cells.set_index(["session", "roi"]).loc[(session, roi)]
    tf_curves, _ = sftf.tf_curves_by_speed(ex_binned[(session, roi)], threshold=TF_SELECT_THRESHOLD, variable="tf", popt=ex_popt)
    plot_curves_with_static(ax_ij, tf_curves, variable="tf", ylabel="ΔF/F" if j == 0 else "", xlabel=None)
    sess_short = session.split("_")[0]
    ax_ij.set_title(f"{label} (ROI {roi})", fontsize=FS_TITLE, pad=2)
fig.text(x_ex_start + w_ex + gap_ex / 2, y_row3 - 0.040, "Temporal frequency (Hz)", ha="center", fontsize=FS_LABEL)

# PANEL K: Example cells SF tuning by running speed (Row 4, left)
fig.text(x_ex_start - 0.030, y_row4 + h_row4 + 0.015, "K", fontweight="bold", fontsize=FS_PANEL, va="bottom")
for j, (session, roi, label) in enumerate(EXAMPLE_CELLS_SPEED):
    ax_jj = fig.add_axes([x_ex_start + j * (w_ex + gap_ex), y_row4, w_ex, h_row4])
    ex_popt = cells.set_index(["session", "roi"]).loc[(session, roi)]
    sf_curves, _ = sftf.tf_curves_by_speed(ex_binned[(session, roi)], threshold=TF_SELECT_THRESHOLD, variable="sf", popt=ex_popt)
    plot_curves_with_static(ax_jj, sf_curves, variable="sf", ylabel="ΔF/F" if j == 0 else "", xlabel=None)
    ax_jj.tick_params(axis="x", rotation=45)
    sess_short = session.split("_")[0]
    ax_jj.set_title(f"{label} (ROI {roi})", fontsize=FS_TITLE, pad=2)
fig.text(x_ex_start + w_ex + gap_ex / 2, y_row4 - 0.050, "Spatial frequency (cpd)", ha="center", fontsize=FS_LABEL)

# Legend handles for TF (Winter) and SF (Cividis) with continuous line for Static
legend_handles_tf = [
    plt.Line2D([0], [0], color="k", ls="-", marker="o", ms=3.6, lw=1.1, label="Static")
] + [
    plt.Line2D([0], [0], color=col, ls="-", marker="o", ms=3.6, lw=1.1, label=lbl)
    for lbl, col in zip(speed_labels, devon_colors)
]
legend_handles_sf = [
    plt.Line2D([0], [0], color="k", ls="-", marker="o", ms=3.6, lw=1.1, label="Static")
] + [
    plt.Line2D([0], [0], color=col, ls="-", marker="o", ms=3.6, lw=1.1, label=lbl)
    for lbl, col in zip(speed_labels, acton_colors)
]

# Placement spanning whole figure width:
# Example cells: x = 0.050 to 0.292 (0.90 to 5.25 cm)
# Legend: x_leg = 0.315 (5.67 to 7.77 cm)
# Population Col 1: x_pop1 = 0.495 (8.91 to 12.26 cm, width 3.35 cm)
# Population Col 2: x_pop2 = 0.755 (13.59 to 16.94 cm, width 3.35 cm)
x_leg = 0.315
w_leg = 2.1 / FIG_W_CM
w_pop = 3.35 / FIG_W_CM
x_pop1 = 0.495
x_pop2 = 0.755

# Legend for TF between I and K (Row 3)
ax_leg_tf = fig.add_axes([x_leg, y_row3, w_leg, h_row3])
ax_leg_tf.axis("off")
ax_leg_tf.legend(
    handles=legend_handles_tf,
    title="Running speed (TF)",
    fontsize=FS_LEGEND,
    title_fontsize=FS_TITLE,
    frameon=False,
    loc="center left",
)

# Legend for SF between J and L (Row 4)
ax_leg_sf = fig.add_axes([x_leg, y_row4, w_leg, h_row4])
ax_leg_sf.axis("off")
ax_leg_sf.legend(
    handles=legend_handles_sf,
    title="Running speed (SF)",
    fontsize=FS_LEGEND,
    title_fontsize=FS_TITLE,
    frameon=False,
    loc="center left",
)

# PANEL L: Population TF shift relative to overall peak (Row 3, Col 1)
ax_k = fig.add_axes([x_pop1, y_row3, w_pop, h_row3])
sftf.plot_heldout_speed_curves_panel(
    ax_k, "tf", heldout_means, mean_peaks, speed_labels, peak_y=None,
    colors=devon_colors,
    norm_to_own_peak=False,
    fontsize_dict={"title": FS_TITLE, "label": FS_LABEL, "tick": FS_TICK, "legend": FS_LEGEND}
)
ax_k.set_title("TF shift (overall peak)", fontsize=FS_TITLE)
ax_k.set_ylabel("ΔF/F / peak", fontsize=FS_LABEL)
ax_k.set_xlabel("TF relative to peak (octaves)", fontsize=FS_LABEL)
ax_k.set_xticks([-4, -2, 0, 2, 4])
ax_k.tick_params(labelsize=FS_TICK)
fig.text(x_pop1 - 0.035, y_row3 + h_row3 + 0.015, "L", fontweight="bold", fontsize=FS_PANEL, va="bottom")

# PANEL N: Population TF shift normalised to own peak (Row 3, Col 2)
ax_m = fig.add_axes([x_pop2, y_row3, w_pop, h_row3])
sftf.plot_heldout_speed_curves_panel(
    ax_m, "tf", heldout_means, mean_peaks, speed_labels, peak_y=None,
    colors=devon_colors,
    norm_to_own_peak=True,
    fontsize_dict={"title": FS_TITLE, "label": FS_LABEL, "tick": FS_TICK, "legend": FS_LEGEND}
)
ax_m.set_title("TF shift (own peak)", fontsize=FS_TITLE)
ax_m.set_ylabel("Normalized response", fontsize=FS_LABEL)
ax_m.set_xlabel("TF relative to peak (octaves)", fontsize=FS_LABEL)
ax_m.set_xticks([-4, -2, 0, 2, 4])
ax_m.tick_params(labelsize=FS_TICK)
fig.text(x_pop2 - 0.035, y_row3 + h_row3 + 0.015, "N", fontweight="bold", fontsize=FS_PANEL, va="bottom")

# PANEL M: Population SF shift relative to overall peak (Row 4, Col 1)
ax_l = fig.add_axes([x_pop1, y_row4, w_pop, h_row4])
sftf.plot_heldout_speed_curves_panel(
    ax_l, "sf", heldout_means, mean_peaks, speed_labels, peak_y=None,
    colors=acton_colors,
    norm_to_own_peak=False,
    fontsize_dict={"title": FS_TITLE, "label": FS_LABEL, "tick": FS_TICK, "legend": FS_LEGEND}
)
ax_l.set_title("SF shift (overall peak)", fontsize=FS_TITLE)
ax_l.set_ylabel("ΔF/F / peak", fontsize=FS_LABEL)
ax_l.set_xlabel("SF relative to peak (octaves)", fontsize=FS_LABEL)
ax_l.set_xticks([-4, -2, 0, 2, 4])
ax_l.tick_params(labelsize=FS_TICK)
fig.text(x_pop1 - 0.035, y_row4 + h_row4 + 0.015, "M", fontweight="bold", fontsize=FS_PANEL, va="bottom")

# PANEL O: Population SF shift normalised to own peak (Row 4, Col 2)
ax_n = fig.add_axes([x_pop2, y_row4, w_pop, h_row4])
sftf.plot_heldout_speed_curves_panel(
    ax_n, "sf", heldout_means, mean_peaks, speed_labels, peak_y=None,
    colors=acton_colors,
    norm_to_own_peak=True,
    fontsize_dict={"title": FS_TITLE, "label": FS_LABEL, "tick": FS_TICK, "legend": FS_LEGEND}
)
ax_n.set_title("SF shift (own peak)", fontsize=FS_TITLE)
ax_n.set_ylabel("Normalized response", fontsize=FS_LABEL)
ax_n.set_xlabel("SF relative to peak (octaves)", fontsize=FS_LABEL)
ax_n.set_xticks([-4, -2, 0, 2, 4])
ax_n.tick_params(labelsize=FS_TICK)
fig.text(x_pop2 - 0.035, y_row4 + h_row4 + 0.015, "O", fontweight="bold", fontsize=FS_PANEL, va="bottom")

# Save outputs
out_pdf = SAVE_ROOT / "fig_sftf_analysis.pdf"
out_png = SAVE_ROOT / "fig_sftf_analysis.png"
style.savefig(out_pdf, fig=fig, bbox_inches="tight")
fig.savefig(out_png, dpi=300, bbox_inches="tight")
print(f"Saved figure to {out_pdf} and {out_png}")


## Overlap of depth-selective and SF/TF-tuned cells (stats only, no panel)

Are depth selectivity and SF/TF tuning carried by separate sub-populations? The count of cells that are both is
compared with the count expected if the two were independent. The fractions of tuned cells differ between sessions,
so the expected count is computed in each session (`n_depth * n_sftf / n`) and summed. The test is a Mantel-Haenszel
test of the pooled odds ratio over the per-session 2 x 2 tables (depth selective x SF/TF tuned).

In [ ]:
# overlap of depth-selective and SF/TF-tuned cells, expected count per session, Mantel-Haenszel test
from statsmodels.stats.contingency_tables import StratifiedTable

overlap_rows, overlap_tables = [], []
for session, sc in cells.groupby("session"):
    n, n_depth, n_sftf = len(sc), int(sc.depth_tuned.sum()), int(sc.sftf_tuned.sum())
    overlap_rows.append(
        dict(
            session=session,
            n=n,
            n_depth=n_depth,
            n_sftf=n_sftf,
            n_both=int((sc.depth_tuned & sc.sftf_tuned).sum()),
            n_both_expected=n_depth * n_sftf / n,
        )
    )
    overlap_tables.append(
        pd.crosstab(sc.depth_tuned, sc.sftf_tuned)
        .reindex(index=[True, False], columns=[True, False], fill_value=0)
        .to_numpy()
    )
overlap_sessions = pd.DataFrame(overlap_rows)
overlap = overlap_sessions.drop(columns="session").sum()
overlap_cmh = StratifiedTable(overlap_tables)
overlap_or = overlap_cmh.oddsratio_pooled
overlap_p = overlap_cmh.test_null_odds().pvalue

print(overlap_sessions.round(1).to_string(index=False))
print(
    f"Depth selective and SF/TF tuned: {overlap.n_both:.0f} / {overlap.n:.0f} cells "
    f"({100 * overlap.n_both / overlap.n:.1f}%); expected if independent within each session: "
    f"{overlap.n_both_expected:.1f} ({100 * overlap.n_both_expected / overlap.n:.1f}%)"
)
print(f"Mantel-Haenszel pooled odds ratio {overlap_or:.2f}, p = {overlap_p:.2g}")


## Export statistics

Every value is read from a variable computed above; nothing is recomputed. Written to
`v1_depth_map/stats/generated/stats_fig_reviewer_gratings.yaml` (group `reviewer`);
`python -m v1_depth_map.stats.build_manuscript_stats` compiles it into the dashboard and the LaTeX macros.

In [ ]:
from v1_depth_map.stats import export_figure_stats


def _fmt_pval(p):
    # p-value as it should read in LaTeX, in math mode so it works in text or math
    p = float(p)
    return "$p < 0.0001$" if p < 1e-4 else f"$p = {p:.3g}$"


def _stat(raw, formatted, macro, description):
    return dict(raw=raw, formatted=formatted, latex_macro=macro, description=description)


def _count(n, macro, description):
    return _stat(int(n), f"{int(n):,}", macro, description)


def _pct(fraction, macro, description):
    return _stat(float(100 * fraction), f"{100 * fraction:.0f}\\%", macro, description)


stats_reviewer = {
    "n_neurons": _count(overlap.n, "statRevGratNeurons", "iscell neurons in the 6 grating sessions"),
    "n_sessions": _count(cells.session.nunique(), "statRevGratSessions", "Sessions with drifting gratings"),
    "n_mice": _count(
        cells.session.str.split("_").str[0].nunique(), "statRevGratMice", "Mice with drifting gratings"
    ),
    "n_depth": _count(overlap.n_depth, "statRevGratDepthNeurons", "Depth-selective neurons"),
    "pct_depth": _pct(overlap.n_depth / overlap.n, "statRevGratPctDepth", "Percentage depth selective"),
    "n_sftf": _count(overlap.n_sftf, "statRevGratSftfNeurons", "SF/TF-tuned neurons, cross-validated"),
    "pct_sftf": _pct(overlap.n_sftf / overlap.n, "statRevGratPctSftf", "Percentage SF/TF tuned"),
    "n_both": _count(overlap.n_both, "statRevGratBothN", "Depth-selective and SF/TF-tuned neurons"),
    "pct_both": _pct(
        overlap.n_both / overlap.n, "statRevGratPctBoth", "Percentage depth selective and SF/TF tuned"
    ),
    "n_both_expected": _stat(
        float(overlap.n_both_expected),
        f"{overlap.n_both_expected:.0f}",
        "statRevGratBothExpectedN",
        "Depth selective and SF/TF tuned expected if independent, computed per session and summed",
    ),
    "pct_both_expected": _pct(
        overlap.n_both_expected / overlap.n,
        "statRevGratPctBothExpected",
        "Percentage both expected if independent, computed per session",
    ),
    "overlap_or": _stat(
        float(overlap_or),
        f"{overlap_or:.2f}",
        "statRevGratOverlapOR",
        "Mantel-Haenszel pooled odds ratio over sessions, depth selective x SF/TF tuned",
    ),
    "overlap_p": _stat(
        float(overlap_p),
        _fmt_pval(overlap_p),
        "statRevGratOverlapP",
        "Mantel-Haenszel test of the pooled odds ratio over sessions",
    ),
}
export_figure_stats("fig_reviewer_gratings", stats_reviewer, figure="reviewer")
